# LangGraph 最小验证 Notebook
---
目标：验证 LangGraph `StateGraph` 基础流转，建立 `GlobalState` 定义与各层节点骨架。

**不包含**：完整 LLM 推理、Primitive 调用、RoboDK 连接。  
**约定**：所有生产逻辑在 `SkiLib/` 中实现，此 Notebook 仅用于框架验证。

In [1]:
# ── Dependencies ──────────────────────────────────────────────────────────────
%pip install -q python-dotenv
%pip install -qU langchain-core langchain-ollama
%pip install -qU langgraph
%pip install -qU langchain-anthropic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# ── Imports & Environment ─────────────────────────────────────────────────────
import os
import getpass
import logging
from dotenv import load_dotenv

from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_anthropic import ChatAnthropic

from langgraph.graph import StateGraph, START, END

# Load API keys / tracing config from .env at project root
load_dotenv()

# LangSmith tracing (optional — set LANGSMITH_TRACING=true in .env to enable)
if os.getenv("LANGSMITH_TRACING", "false").lower() == "true":
    if not os.getenv("LANGSMITH_API_KEY"):
        os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter LangSmith API Key: ")
    if not os.getenv("LANGSMITH_PROJECT"):
        os.environ["LANGSMITH_PROJECT"] = getpass.getpass("Enter LangSmith Project name: ")

logging.basicConfig(level=logging.ERROR, force=True)
print("Environment loaded. LangSmith tracing:", os.getenv("LANGSMITH_TRACING", "false"))

Environment loaded. LangSmith tracing: true


In [ ]:
LLM_TYPE = "claude" # choose from "claude" and "ollama" (local)
# ── LLM Configuration ─────────────────────────────────────────────────────────
# Uses local Ollama. Change OLLAMA_MODEL_ID to switch models.
OLLAMA_MODEL_ID = os.getenv("OLLAMA_MODEL_ID", "qwen3:latest")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

if LLM_TYPE == "claude":
    llm = ChatAnthropic(
        model="claude-sonnet-4-6"
    )
else:
    llm = ChatOllama(
        model=OLLAMA_MODEL_ID,
        base_url=OLLAMA_BASE_URL,
        temperature=0,
    )



# Quick connectivity check
try:
    _ping = llm.invoke("Reply with one word: ready")
    print("LLM reachable:", _ping.content.strip()[:80])
except Exception as e:
    print(f"[WARN] LLM not reachable — nodes using llm will fail. ({e})")

## GlobalState
与 `CLAUDE.md` 中 `GlobalState` 规范保持一致。  
`robot_state` 此处用 `dict` 代替，生产版本从 `SkiLib.base` 导入 `RobotState`。

In [ ]:
# ── GlobalState Definition ────────────────────────────────────────────────────
from typing import TypedDict, Annotated, Optional
import operator


class GlobalState(TypedDict):
    # Layer-1: planning outputs
    todo_list: list[dict]           # Planner generates [{task_id, skill, params}, ...]
    # Layer-2: execution context
    current_task: dict              # Dispatcher peeks todo_list[0]; NOT popped yet
    # Robot runtime snapshot (stub — production type: SkiLib.base.RobotState)
    robot_state: dict
    # Control flags
    halt_flag: bool                 # True = all R-skill execution locked
    # Executor writes result here; Context Flush pops todo_list only on success, then clears
    last_result: Optional[dict]
    # Audit trail written by Context Flush; Annotated list avoids key overwrite
    execution_log: Annotated[list[str], operator.add]
    # LangGraph message bus
    messages: Annotated[list[BaseMessage], operator.add]


print("GlobalState keys:", list(GlobalState.__annotations__.keys()))

## Node Stubs
每个节点只做最小操作（打印 + 返回 state 片段），验证图的流转逻辑。  
完整实现迁移至 `SkiLib/` 后替换对应函数体即可。

| 节点 | 层 | 当前状态 |
|------|----|----------|
| `supervisor` | Layer-1 | STUB — 直接透传消息 |
| `planner` | Layer-1 | STUB — 生成硬编码 `todo_list` |
| `dispatcher` | Layer-2 | 已实现 — 纯代码 peek `todo_list[0]`（不 pop） |
| `executor` | Layer-2 | STUB — 模拟执行成功，写 `last_result` |
| `context_flush` | Layer-2 | 已实现 — 成功时 pop `todo_list`，失败时保留任务触发 halt |

In [ ]:
# ── Node: Supervisor ──────────────────────────────────────────────────────────
# Role: gather domain knowledge, resolve ambiguities via task-skills.
# Rule: never compute coordinates; only handle symbols (Target_A, Tool_Gripper).
# TODO: replace stub with real ReAct loop + task-skills
def supervisor(state: GlobalState) -> dict:
    print("[supervisor] Analyzing instruction...")
    last_msg = state["messages"][-1].content if state["messages"] else "(no input)"
    return {
        "messages": [AIMessage(content=f"[Supervisor] Instruction understood: {last_msg}")]
    }


# ── Node: Planner ─────────────────────────────────────────────────────────────
# Role: emit structured todo_list JSON via forced structured output.
# Rule: output must be valid JSON; add schema validation + retry in production.
# TODO: replace stub with LLM structured output + Pydantic schema + retry
def planner(state: GlobalState) -> dict:
    print("[planner] Generating task plan...")
    todo = [
        {"task_id": "t1", "skill": "MoveJ",       "params": {"target": "Home"}},
        {"task_id": "t2", "skill": "PickAndPlace", "params": {"pick": "Part_A", "place": "Tray_1"}},
        {"task_id": "t3", "skill": "MoveJ",       "params": {"target": "Home"}},
    ]
    return {
        "todo_list": todo,
        "messages":  [AIMessage(content=f"[Planner] {len(todo)} tasks queued")],
    }


# ── Node: Dispatcher (pure code, no LLM) ─────────────────────────────────────
# Role: fill the execution slot — pop todo_list[0] into current_task ONLY when slot is empty.
#       If current_task is already set (e.g. retained after failure), do nothing.
# Rule: 100% deterministic; must never invoke any LLM.
def dispatcher(state: GlobalState) -> dict:
    if state.get("current_task"):
        # Slot occupied — task in flight or retained after failure; do not overwrite
        print(f"[dispatcher] Slot occupied: {state['current_task'].get('task_id')} — skipping pop.")
        return {}
    todo = list(state.get("todo_list", []))
    if not todo:
        print("[dispatcher] todo_list empty — nothing to dispatch.")
        return {}
    task = todo.pop(0)
    print(f"[dispatcher] Dispatching: {task}")
    return {"current_task": task, "todo_list": todo}


# ── Node: Executor ────────────────────────────────────────────────────────────
# Role: execute current_task via the matching Skill; report result in last_result.
# Rule: @require_robot_active must guard all R-skills; halt_flag checked here.
# TODO: replace stub with dynamic Skill loader + SkiLib.base.SkillResult
def executor(state: GlobalState) -> dict:
    task = state.get("current_task", {})
    if not task:
        return {
            "execution_log": ["[executor] No task — skipping."],
            "last_result": {"success": True},
        }
    if state.get("halt_flag"):
        return {
            "execution_log": [f"[executor] HALTED — skipping {task.get('task_id')}"],
            "last_result": {"success": False, "error_type": "ROBOT_INACTIVE"},
        }
    print(f"[executor] Running: {task['skill']}({task['params']})")
    # STUB: simulate success without touching robot
    return {
        "execution_log": [f"[executor] {task['task_id']} {task['skill']} -> SUCCESS (stub)"],
        "last_result": {"success": True},
    }


# ── Node: Context Flush (pure code) ──────────────────────────────────────────
# Role: on success — clear current_task (empty the slot) and clear last_result.
#       on failure — set halt_flag; current_task and todo_list are left intact so
#                    the same task will be retried after human intervention resumes the system.
# TODO: add RemoveMessage sweep once Executor uses real LangGraph tool calls
def context_flush(state: GlobalState) -> dict:
    task_id = state.get("current_task", {}).get("task_id", "?")
    last_result = state.get("last_result") or {}

    if last_result.get("success"):
        entry = f"[context_flush] Task {task_id} committed."
        print(entry)
        return {
            "current_task": {},    # empty the slot — Dispatcher will fill it next iteration
            "last_result":  None,
            "execution_log": [entry],
        }
    else:
        entry = f"[context_flush] Task {task_id} FAILED — halting, task retained in queue."
        print(entry)
        return {
            "halt_flag":     True,  # lock the system; current_task stays for retry after resume
            "execution_log": [entry],
        }


print("All node stubs defined.")

## Graph Construction
流转顺序：`START → supervisor → planner → [dispatcher → executor → context_flush] × N → END`

`should_continue` 路由逻辑（优先级从高到低）：

| 条件 | 路由 | 说明 |
|------|------|------|
| `halt_flag=True` | `halt → END` | 执行失败或人工介入请求；`current_task` 保留，resume 后重试 |
| `todo_list` 非空 | `continue → dispatcher` | 槽已清空，取下一个任务 |
| 其他 | `done → END` | 队列清空，正常结束 |

**`current_task` 作为执行槽的状态语义：**
- `{}` → 槽空闲，Dispatcher 负责填充
- `{...}` → 任务在执行中（或失败保留），Dispatcher 跳过不覆盖

In [ ]:
# ── Graph Construction ────────────────────────────────────────────────────────

def should_continue(state: GlobalState) -> str:
    """Routing condition after context_flush. Priority: halt > continue > done."""
    if state.get("halt_flag"):
        return "halt"
    if state.get("todo_list"):
        return "continue"
    return "done"


builder = StateGraph(GlobalState)

builder.add_node("supervisor",    supervisor)
builder.add_node("planner",       planner)
builder.add_node("dispatcher",    dispatcher)
builder.add_node("executor",      executor)
builder.add_node("context_flush", context_flush)

# Layer-1: linear
builder.add_edge(START,        "supervisor")
builder.add_edge("supervisor", "planner")
builder.add_edge("planner",    "dispatcher")

# Layer-2: execution loop
builder.add_edge("dispatcher",   "executor")
builder.add_edge("executor",     "context_flush")
builder.add_conditional_edges(
    "context_flush",
    should_continue,
    {
        "continue": "dispatcher",  # slot cleared — fetch next task
        "done":     END,           # queue empty — all done
        "halt":     END,           # halt_flag set — await human intervention
    },
)

graph = builder.compile()
print("Graph compiled.")

In [ ]:
# Optional: visualize graph (requires graphviz)
try:
    from IPython.display import Image
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Visualization skipped:", e)
    print(graph.get_graph().draw_mermaid())

## Test Run

In [ ]:
# ── Test Invocation ───────────────────────────────────────────────────────────
initial_state: GlobalState = {
    "messages":      [HumanMessage(content="将 Part_A 放入 Tray_1")],
    "todo_list":     [],
    "current_task":  {},
    "robot_state":   {"joints": None, "pose": None, "gripper_state": "UNKNOWN"},
    "halt_flag":     False,
    "last_result":   None,
    "execution_log": [],
}

print("=" * 60)
final_state = graph.invoke(initial_state)
print("=" * 60)
print("Execution log:")
for entry in final_state["execution_log"]:
    print(" ", entry)
print("Remaining todo_list:", final_state["todo_list"])
print("halt_flag:",           final_state["halt_flag"])
print("last_result:",         final_state["last_result"])